In [ ]:
# === Cell unique: EVAL VAL/TEST + choix de seuil sur VAL via EV = audit_R_bps ===
# Hypothèses:
# - StageB parquet contient: label_A, features (feature_cols_xgb), audit_R_bps, audit_p_thr_ev0
# - Modèle XGB est un JSON (booster.save_model(...)) sur S3 ou local
#
# Outputs:
# - best threshold thr* choisi sur VAL (max sum(audit_R_bps) sous contrainte n_trades>=MIN_TRADES)
# - report dict imprimé + optionnel upload S3 (JSON)

import json, time, math
import numpy as np
import pandas as pd
import s3fs
import xgboost as xgb

# --------------------------
# CONFIG (ajuste si besoin)
# --------------------------
STAGEB_ROOT = "s3://tradebot-config-tokyo/data/stageB/dataset=v1"
COLUMNS_JSON = "s3://tradebot-config-tokyo/data/stageB/dataset=v1/_meta/columns.json"
ARTIFACT_KEY = "stageB_parquet"  # "stageB_parquet" ou "stageB_csv_gz" (ici on veut audit_*, donc parquet)

MODEL_URI = "s3://tradebot-config-tokyo/models/xgb-baseline/stageB/baseline_xgb_stageB_20260203-100728.json"
# ou local: MODEL_URI = "/tmp/baseline_xgb_stageB_20260203-100728.json"

# lecture (rapide): nb de fichiers par split (0 => tous)
VAL_FILES  = 0
TEST_FILES = 0

# cap lignes après concat (None => pas de cap)
VAL_MAX_ROWS  = None
TEST_MAX_ROWS = None

SEED = 42

# contrainte stabilité (pas un objectif de fréquence)
MIN_TRADES = 200

# grid thresholds (quantiles des proba)
THR_GRID_QUANTILES = np.r_[np.linspace(0.90, 0.99, 10), np.linspace(0.991, 0.999, 9), [0.9995, 0.9997, 0.9999]]
# --------------------------

fs = s3fs.S3FileSystem()

def read_json_s3(uri: str) -> dict:
    with fs.open(uri, "rb") as f:
        return json.load(f)

def list_stageb_paths_by_split(stageb_root: str, split: str, artifact_key: str):
    if artifact_key == "stageB_parquet":
        sub, suffix = "parquet", ".parquet"
    else:
        sub, suffix = "xgb", ".csv.gz"
    prefix = f"{stageb_root}/split={split}/{sub}"
    pattern = prefix.replace("s3://", "") + f"/*{suffix}"
    paths = [f"s3://{p}" for p in fs.glob(pattern)]
    if not paths:
        raise FileNotFoundError(f"No StageB files for split={split} under {prefix}")
    return sorted(paths)

def sample_paths(paths, n_files, seed):
    if n_files <= 0 or n_files >= len(paths):
        return paths
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(paths), size=n_files, replace=False)
    return [paths[i] for i in idx]

def load_split_df(paths, cols, max_rows=None, seed=0):
    dfs = []
    for p in paths:
        if p.endswith(".parquet"):
            dfs.append(pd.read_parquet(p, columns=cols, engine="pyarrow"))
        else:
            dfs.append(pd.read_csv(p, compression="gzip", usecols=cols))
    df = pd.concat(dfs, ignore_index=True)
    if max_rows is not None and max_rows > 0 and len(df) > max_rows:
        df = df.sample(n=int(max_rows), random_state=int(seed)).reset_index(drop=True)
    return df

def load_booster(uri: str) -> xgb.Booster:
    b = xgb.Booster()
    if uri.startswith("s3://"):
        with fs.open(uri, "rb") as f:
            raw = f.read()
        b.load_model(bytearray(raw))
    else:
        b.load_model(uri)
    return b

def eval_at_thr(p, y, R_bps, p_thr_ev0, thr):
    p = np.asarray(p, float)
    y = np.asarray(y, int)
    R = np.asarray(R_bps, float)
    pt0 = np.asarray(p_thr_ev0, float)

    # gates: decision threshold + EV>=0 threshold
    m = (p >= float(thr)) & (p >= pt0)

    n = int(m.sum())
    if n == 0:
        return {
            "thr": float(thr),
            "n_trades": 0,
            "sum_R_bps": 0.0,
            "mean_R_bps": 0.0,
            "hit_rate_Rpos": 0.0,
            "precision_label": 0.0,
            "pos_rate_in_trades": 0.0,
            "mean_margin_p_minus_pthr": 0.0,
            "mean_p": 0.0,
            "mean_pthr_ev0": 0.0,
        }

    Rt = R[m]
    yt = y[m]
    pt = p[m]
    pt0t = pt0[m]

    sumR = float(np.sum(Rt))
    meanR = float(np.mean(Rt))
    hit = float(np.mean(Rt > 0))
    prec = float(np.mean(yt == 1))  # proxy precision label

    margin = float(np.mean(pt - pt0t)) if pt0t.size else float("nan")

    return {
        "thr": float(thr),
        "n_trades": n,
        "sum_R_bps": sumR,
        "mean_R_bps": meanR,
        "hit_rate_Rpos": hit,
        "precision_label": prec,
        "pos_rate_in_trades": prec,
        "mean_margin_p_minus_pthr": margin,
        "mean_p": float(np.mean(pt)),
        "mean_pthr_ev0": float(np.mean(pt0t)),
    }

# ---- meta columns
meta = read_json_s3(COLUMNS_JSON)
LABEL_COL = meta.get("label_col_for_csv") or meta.get("label_col") or "label_A"
FEATURE_COLS = meta.get("feature_cols_xgb") or meta.get("feature_cols_all")
if not isinstance(FEATURE_COLS, list) or len(FEATURE_COLS) == 0:
    raise RuntimeError("No feature cols in columns.json (feature_cols_xgb/feature_cols_all missing)")

# audits requis
AUDIT_R = "audit_R_bps"
AUDIT_PTHR = "audit_p_thr_ev0"

# ---- paths
val_paths_all  = list_stageb_paths_by_split(STAGEB_ROOT, "val", ARTIFACT_KEY)
test_paths_all = list_stageb_paths_by_split(STAGEB_ROOT, "test", ARTIFACT_KEY)

val_paths  = sample_paths(val_paths_all,  int(VAL_FILES),  SEED + 1)
test_paths = sample_paths(test_paths_all, int(TEST_FILES), SEED + 2)

# ---- load dfs (VAL/TEST) avec features + audits + label
cols_needed = [LABEL_COL, AUDIT_R, AUDIT_PTHR] + FEATURE_COLS
df_val  = load_split_df(val_paths,  cols_needed, max_rows=VAL_MAX_ROWS,  seed=SEED + 11)
df_test = load_split_df(test_paths, cols_needed, max_rows=TEST_MAX_ROWS, seed=SEED + 12)

# ---- basic sanity
for where, df in [("VAL", df_val), ("TEST", df_test)]:
    u = pd.unique(df[LABEL_COL])
    if not np.all(np.isin(u, [0,1])):
        raise RuntimeError(f"[{where}] label not binary; uniques={u[:10]}")
    if df[AUDIT_R].isna().any():
        raise RuntimeError(f"[{where}] {AUDIT_R} has NaN (cannot optimize EV safely)")
    if df[AUDIT_PTHR].isna().any():
        raise RuntimeError(f"[{where}] {AUDIT_PTHR} has NaN (sanity gate becomes meaningless)")
    Xtmp = df[FEATURE_COLS].to_numpy(dtype="float64", copy=False)
    if np.isinf(Xtmp).any():
        raise RuntimeError(f"[{where}] found +/-inf in features")

# ---- load model + predict
booster = load_booster(MODEL_URI)

Xv = df_val[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)
Xt = df_test[FEATURE_COLS].to_numpy(dtype=np.float32, copy=False)

pv = booster.predict(xgb.DMatrix(Xv, feature_names=FEATURE_COLS), output_margin=False)
pt = booster.predict(xgb.DMatrix(Xt, feature_names=FEATURE_COLS), output_margin=False)

if not np.isfinite(pv).all() or not np.isfinite(pt).all():
    raise RuntimeError("Model produced NaN/inf probabilities")
if pv.min() < -1e-6 or pv.max() > 1.0 + 1e-6:
    raise RuntimeError(f"VAL proba out of [0,1]: min={pv.min()} max={pv.max()}")
if pt.min() < -1e-6 or pt.max() > 1.0 + 1e-6:
    raise RuntimeError(f"TEST proba out of [0,1]: min={pt.min()} max={pt.max()}")

# ---- threshold grid from VAL quantiles
qs = THR_GRID_QUANTILES[(THR_GRID_QUANTILES > 0) & (THR_GRID_QUANTILES < 1)]
thr_grid = np.unique(np.quantile(pv, qs))
thr_grid = np.clip(thr_grid, 0.0, 1.0)

# add a couple of safe extremes
thr_grid = np.unique(np.r_[thr_grid, [0.0, 1.0]])

# ---- sweep on VAL: maximize sum_R_bps with MIN_TRADES
y_val = df_val[LABEL_COL].to_numpy(np.int32, copy=False)
R_val = df_val[AUDIT_R].to_numpy(np.float64, copy=False)
pthr_val = df_val[AUDIT_PTHR].to_numpy(np.float64, copy=False)

best = None
rows = []
for thr in thr_grid:
    r = eval_at_thr(pv, y_val, R_val, pthr_val, float(thr))
    rows.append(r)
    if r["n_trades"] < int(MIN_TRADES):
        continue
    if best is None:
        best = r
    else:
        # primary: sum_R_bps, tie: higher mean_R_bps, tie: more trades, tie: higher thr (more conservative)
        if (
            r["sum_R_bps"] > best["sum_R_bps"] + 1e-12 or
            (abs(r["sum_R_bps"] - best["sum_R_bps"]) <= 1e-12 and r["mean_R_bps"] > best["mean_R_bps"] + 1e-12) or
            (abs(r["sum_R_bps"] - best["sum_R_bps"]) <= 1e-12 and abs(r["mean_R_bps"] - best["mean_R_bps"]) <= 1e-12 and r["n_trades"] > best["n_trades"]) or
            (abs(r["sum_R_bps"] - best["sum_R_bps"]) <= 1e-12 and abs(r["mean_R_bps"] - best["mean_R_bps"]) <= 1e-12 and r["n_trades"] == best["n_trades"] and r["thr"] > best["thr"])
        ):
            best = r

if best is None:
    # fallback: choose thr that gives max sum_R_bps even if low trades (and warn)
    best = max(rows, key=lambda z: (z["sum_R_bps"], z["mean_R_bps"], z["n_trades"], z["thr"]))
    note = f"WARNING: no threshold met MIN_TRADES={MIN_TRADES}. Using best by sum_R_bps without constraint."
else:
    note = "ok"

# ---- evaluate chosen thr on TEST
y_test = df_test[LABEL_COL].to_numpy(np.int32, copy=False)
R_test = df_test[AUDIT_R].to_numpy(np.float64, copy=False)
pthr_test = df_test[AUDIT_PTHR].to_numpy(np.float64, copy=False)

val_best  = eval_at_thr(pv, y_val,  R_val,  pthr_val,  best["thr"])
test_best = eval_at_thr(pt, y_test, R_test, pthr_test, best["thr"])

stamp = time.strftime("%Y%m%d-%H%M%S")
report = {
    "timestamp": stamp,
    "note": note,
    "model_uri": MODEL_URI,
    "stageb_root": STAGEB_ROOT,
    "columns_json": COLUMNS_JSON,
    "artifact_key": ARTIFACT_KEY,
    "label_col": LABEL_COL,
    "n_features": int(len(FEATURE_COLS)),
    "min_trades_constraint": int(MIN_TRADES),
    "threshold_policy": {
        "chosen_on": "VAL only",
        "objective": "maximize sum(audit_R_bps) over trades where p>=thr",
        "thr_grid_quantiles": [float(x) for x in qs.tolist()],
        "thr_chosen": float(best["thr"]),
    },
    "val": {
        "n_rows": int(len(df_val)),
        "pos": int((y_val == 1).sum()),
        "neg": int((y_val == 0).sum()),
        "pos_rate": float((y_val == 1).mean()),
        "best": val_best,
        "score_stats": {
            "p_mean": float(np.mean(pv)),
            "p_p50": float(np.quantile(pv, 0.50)),
            "p_p95": float(np.quantile(pv, 0.95)),
            "p_p99": float(np.quantile(pv, 0.99)),
            "p_max": float(np.max(pv)),
        },
    },
    "test": {
        "n_rows": int(len(df_test)),
        "pos": int((y_test == 1).sum()),
        "neg": int((y_test == 0).sum()),
        "pos_rate": float((y_test == 1).mean()),
        "best_at_val_thr": test_best,
        "score_stats": {
            "p_mean": float(np.mean(pt)),
            "p_p50": float(np.quantile(pt, 0.50)),
            "p_p95": float(np.quantile(pt, 0.95)),
            "p_p99": float(np.quantile(pt, 0.99)),
            "p_max": float(np.max(pt)),
        },
    },
}

print(json.dumps(report, indent=2, ensure_ascii=False))

# OPTIONAL: write report to S3
OUT_S3 = f"s3://tradebot-config-tokyo/models/xgb-baseline/stageB/eval_ev_stageB_{stamp}.json"
with fs.open(OUT_S3, "wb") as f:
    f.write(json.dumps(report, indent=2, ensure_ascii=False).encode("utf-8"))
print(f"\n[save] wrote report -> {OUT_S3}")